In [1]:
!pip install -q transformers datasets accelerate scikit-learn langdetect

import os
import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Usando dispositivo:", device)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Usando dispositivo: cuda


In [2]:
train_df = pd.read_csv('train.csv', encoding='utf-8')
eval_df = pd.read_csv('eval.csv', encoding='utf-8')

label2id = {'negativo': 0, 'neutral': 1, 'positivo': 2}
id2label = {0: 'negativo', 1: 'neutral', 2: 'positivo'}
train_df['target'] = train_df['label'].map(label2id)

# Detector rápido de idioma para estratificación y análisis de errores por subgrupo
def detect_lang(text):
    t = text.lower()
    words = set(re.findall(r'\b[a-záéíóúñãõçâêîôûàèù]+\b', t))
    es = len(words & {'el','la','los','las','que','en','un','una','por','para','con','pero','muy','lo','me','se','del','al','es','son','fue','está','más','también','cuando','todo','esta','este','bien','malo','bueno','llegó','compré','pedí','uso','días','casa'})
    en = len(words & {'the','and','to','of','is','in','it','for','that','with','on','was','but','not','are','have','this','my','very','good','great','bad','product','bought','use','days','quality','battery','would','after'})
    pt = len(words & {'não','com','uma','para','por','mais','como','mas','muito','bem','bom','boa','ruim','comprei','chegou','dias','casa','uso','também','quando','isso','essa','esse','são','está','fazer','qualidade','bateria','nas','dos','das','pelo','pela'})
    fr = len(words & {'le','la','les','des','est','une','dans','pour','pas','que','avec','sur','mais','plus','très','bien','bon','mauvais','acheté','livraison','jours','maison','qualité','batterie','depuis','tout','cette','sont','fait','avoir'})
    if es >= 3 and en >= 3:
        return 'es-en'
    scores = {'es': es, 'en': en, 'pt': pt, 'fr': fr}
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'other'

train_df['lang'] = train_df['text'].apply(detect_lang)
eval_df['lang'] = eval_df['text'].apply(detect_lang)

# Estratificación conjunta: clase + idioma (85% train, 15% val)
train_df['strat_key'] = train_df['label'] + '_' + train_df['lang']
skf = StratifiedKFold(n_splits=6, shuffle=True, random_state=42)
for train_idx, val_idx in skf.split(train_df, train_df['strat_key']):
    trn_df = train_df.iloc[train_idx].reset_index(drop=True)
    val_df = train_df.iloc[val_idx].reset_index(drop=True)
    break

print(f"Train size: {len(trn_df)} | Val size: {len(val_df)}")

Train size: 17055 | Val size: 3411


/usr/local/lib/python3.13/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=6.
  warnings.warn(


In [3]:
MODEL_NAME = 'xlm-roberta-base'
MAX_LEN = 192
BATCH_SIZE = 32
EPOCHS = 4
LR = 2e-5

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ReviewDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            padding='max_length',
            max_length=MAX_LEN,
            return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long) if labels is not None else None

    def __len__(self):
        return len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        item = {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx]
        }
        if self.labels is not None:
            item['labels'] = self.labels[idx]
        return item

trn_ds = ReviewDataset(trn_df['text'], trn_df['target'].values)
val_ds = ReviewDataset(val_df['text'], val_df['target'].values)
eval_ds = ReviewDataset(eval_df['text'])

trn_loader = DataLoader(trn_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False)
eval_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE * 2, shuffle=False)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(trn_loader) * EPOCHS
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

best_val_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    for batch in trn_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()

    # Validación
    model.eval()
    val_preds, val_targets = [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_targets.extend(batch['labels'].numpy())

    val_acc = accuracy_score(val_targets, val_preds)
    print(f"Época {epoch+1}/{EPOCHS} | Train Loss: {total_loss/len(trn_loader):.4f} | Val Accuracy: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_xlmr_baseline.pt')
        val_df['pred'] = val_preds

print(f"\nMejor Val Accuracy (Sin Aumentación): {best_val_acc:.4f}")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

... (5 outputs left out)

/tmp/ipykernel_793/47339176.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Época 3/4 | Train Loss: 0.1439 | Val Accuracy: 0.9033


/tmp/ipykernel_793/47339176.py:57: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):


Época 4/4 | Train Loss: 0.1216 | Val Accuracy: 0.9030

Mejor Val Accuracy (Sin Aumentación): 0.9033


In [4]:
print("=== Reporte Global en Validación ===")
print(classification_report(val_df['target'], val_df['pred'], target_names=['negativo', 'neutral', 'positivo'], digits=4))

print("=== Accuracy por Idioma en Validación ===")
for lang, grp in val_df.groupby('lang'):
    acc_l = accuracy_score(grp['target'], grp['pred'])
    print(f"Idioma {lang:6s} (n={len(grp):4d}): Accuracy = {acc_l:.4f}")

# Cargar mejor checkpoint y predecir sobre eval.csv
model.load_state_dict(torch.load('best_xlmr_baseline.pt'))
model.eval()
eval_preds = []

with torch.no_grad():
    for batch in eval_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        eval_preds.extend(preds)

submission = pd.DataFrame({
    'id': eval_df['id'],
    'answer': [id2label[p] for p in eval_preds]
})

submission.to_csv('submission_1_xlmr_base.csv', index=False)
print("\nArchivo generado: submission_1_xlmr_base.csv")
print("Distribución de predicciones en eval:")
print(submission['answer'].value_counts(normalize=True).round(4))

=== Reporte Global en Validación ===
              precision    recall  f1-score   support

    negativo     0.8764    0.8528    0.8644      1230
     neutral     1.0000    0.9968    0.9984       932
    positivo     0.8584    0.8831    0.8706      1249

    accuracy                         0.9033      3411
   macro avg     0.9116    0.9109    0.9111      3411
weighted avg     0.9036    0.9033    0.9033      3411

=== Accuracy por Idioma en Validación ===
Idioma en     (n= 391): Accuracy = 0.8772
Idioma es     (n=2282): Accuracy = 0.9115
Idioma es-en  (n=  22): Accuracy = 0.6364
Idioma fr     (n= 346): Accuracy = 0.8902
Idioma other  (n=   1): Accuracy = 1.0000
Idioma pt     (n= 369): Accuracy = 0.9079

Archivo generado: submission_1_xlmr_base.csv
Distribución de predicciones en eval:
answer
positivo    0.4330
negativo    0.3755
neutral     0.1916
Name: proportion, dtype: float64
